In [1]:
!pip install ultralytics --upgrade
!pip install torch torchvision torchaudio
!pip install opencv-python numpy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 922.2/922.2 kB 22.1 MB/s eta 0:00:0000:01


In [2]:
!git clone https://github.com/aromalgigi96/Cosmic_Navigators.git


Cloning into 'Cosmic_Navigators'...
remote: Enumerating objects: 145234, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 145234 (delta 4), reused 2 (delta 2), pack-reused 145226 (from 4)
Receiving objects: 100% (145234/145234), 2.74 GiB | 48.02 MiB/s, done.
Resolving deltas: 100% (25828/25828), done.
Updating files: 100% (67004/67004), done.


In [3]:
!ls -lh Cosmic_Navigators/dataset/


total 32K
-rw-r--r-- 1 root root  353 Mar  4 04:01 data.yaml
-rw-r--r-- 1 root root  167 Mar  4 04:01 README.dataset.txt
-rw-r--r-- 1 root root 1.3K Mar  4 04:01 README.roboflow.txt
drwxr-xr-x 4 root root 4.0K Mar  4 04:01 test
drwxr-xr-x 4 root root 4.0K Mar  4 04:01 train
drwxr-xr-x 4 root root 4.0K Mar  4 04:01 train_augmented
drwxr-xr-x 4 root root 4.0K Mar  4 04:01 valid
drwxr-xr-x 4 root root 4.0K Mar  4 04:01 valid_augmented


In [4]:
!cat Cosmic_Navigators/dataset/data.yaml

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 5
names: ['large_debris', 'medium_debris', 'rocket', 'satellite', 'small_debris']

roboflow:
  workspace: nsst3gp-9l3x6
  project: space-debris-detection-bxlp3
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/nsst3gp-9l3x6/space-debris-detection-bxlp3/dataset/1

In [5]:
import os

# Check if the dataset exists
dataset_path = "/kaggle/working/Cosmic_Navigators/dataset/data.yaml"

if os.path.exists(dataset_path):
    print("✅ Dataset found!")
else:
    print("❌ Dataset NOT found! Check the path or re-clone the repo.")


✅ Dataset found!


In [6]:
import os
print(os.listdir("/kaggle/working/"))
print(os.listdir("/kaggle/input/"))  # If dataset is in Kaggle datasets


['.virtual_documents', 'Cosmic_Navigators']
[]


In [ ]:
import yaml


correct_data = {
    "train": "/kaggle/working/Cosmic_Navigators/dataset/train_augmented/images",
    "val": "/kaggle/working/Cosmic_Navigators/dataset/valid_augmented/images",
    "test": "/kaggle/working/Cosmic_Navigators/dataset/test/images",  
    "nc": 5,
    "names": ["large_debris", "medium_debris", "rocket", "satellite", "small_debris"]
}


yaml_path = "/kaggle/working/Cosmic_Navigators/dataset/data.yaml"


with open(yaml_path, "w") as f:
    yaml.dump(correct_data, f, default_flow_style=False)

print("✅ data.yaml updated successfully!")


✅ data.yaml updated successfully!


In [8]:
cat /kaggle/working/Cosmic_Navigators/dataset/data.yaml


names:
- large_debris
- medium_debris
- rocket
- satellite
- small_debris
nc: 5
test: /kaggle/working/Cosmic_Navigators/dataset/test/images
train: /kaggle/working/Cosmic_Navigators/dataset/train_augmented/images
val: /kaggle/working/Cosmic_Navigators/dataset/valid_augmented/images


In [ ]:
from ultralytics import YOLO
import yaml
import os


data_config_path = "/kaggle/working/Cosmic_Navigators/dataset/data.yaml"  # Adjust if needed
with open(data_config_path, 'r') as file:
    data_config = yaml.safe_load(file)

print("Data configuration loaded successfully:")
print(data_config)


# Specify the checkpoint path from a previous run (if available)
checkpoint_path = "/kaggle/working/Cosmic_Navigators/last (1).pt"

if os.path.exists(checkpoint_path):
    print(f"Checkpoint found at {checkpoint_path}. Resuming training.")
    model = YOLO(checkpoint_path)  # Load from checkpoint
    resume_flag = True
else:
    print("No checkpoint found. Starting training from scratch.")
    model = YOLO("yolov8m.yaml")   # Start with a fresh YOLOv8m model
    resume_flag = False


results = model.train(
    data=data_config_path,
    epochs=55,       # Train for 100 epochs; adjust as needed
    imgsz=640,        # Input image size (640x640)
    batch=16,         # Batch size of 16; if you encounter memory issues, try reducing this value
    device=0,         # Use GPU 0 (a single GPU)
    project="runs/train",
    name="yolov8_single_gpu",
    resume=resume_flag
)

print("Training completed!")


model.export(format="onnx")
print("Model export completed! Check the export directory for the exported model.")


# (Optional) Check GPU Usage

!nvidia-smi


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Data configuration loaded successfully:
{'names': ['large_debris', 'medium_debris', 'rocket', 'satellite', 'small_debris'], 'nc': 5, 'test': '/kaggle/working/Cosmic_Navigators/dataset/test/images', 'train': '/kaggle/working/Cosmic_Navigators/dataset/train_augmented/images', 'val': '/kaggle/working/Cosmic_Navigators/dataset/valid_augmented/images'}
No checkpoint found. Starting training from scratch.
Ultralytics 8.3.82 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.yaml, data=/kaggle/working/Cosmic_Navigators/dataset/data.yaml, epochs=55, time=None, patience=100, batch=16, imgsz=640, save=True, save_

100%|██████████| 755k/755k [00:00<00:00, 25.9MB/s]


Overriding model.yaml nc=80 with nc=5

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

100%|██████████| 5.35M/5.35M [00:00<00:00, 91.8MB/s]


AMP: checks passed ✅


...


55 epochs completed in 9.278 hours.
Optimizer stripped from runs/train/yolov8_single_gpu/weights/last.pt, 52.0MB
Optimizer stripped from runs/train/yolov8_single_gpu/weights/best.pt, 52.0MB

Validating runs/train/yolov8_single_gpu/weights/best.pt...
Ultralytics 8.3.82 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8m summary (fused): 92 layers, 25,842,655 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 41/41 [00:18<00:00,  2.16it/s]


                   all       1287       8254      0.318       0.42      0.363      0.219
          large_debris        286       7286      0.465      0.764      0.652      0.323
         medium_debris         23         55     0.0813        0.2     0.0656     0.0314
                rocket          8         11          0          0     0.0163    0.00695
             satellite        849        855      0.859      0.968      0.966      0.699
          small_debris          6         47      0.184       0.17      0.117      0.033


/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.2ms preprocess, 10.4ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs/train/yolov8_single_gpu
Training completed!
Ultralytics 8.3.82 🚀 Python-3.10.12 torch-2.5.1+cu121 CPU (Intel Xeon 2.00GHz)
YOLOv8m summary (fused): 92 layers, 25,842,655 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'runs/train/yolov8_single_gpu/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (49.6 MB)
requirements: Ultralytics requirements ['onnxslim', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 283.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 250.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 288.6 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 11.5s, installed 2 packages: ['onnxslim', 'onnxruntime-gpu']
requireme

In [15]:
from ultralytics import YOLO
import yaml
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [17]:
metrics = model.val(data="/kaggle/working/Cosmic_Navigators/dataset/data.yaml", split="test")
print(metrics)


Ultralytics 8.3.82 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 92 layers, 25,842,655 parameters, 0 gradients, 78.7 GFLOPs


val: Scanning /kaggle/working/Cosmic_Navigators/dataset/test/labels.cache... 649 images, 1 backgrounds, 0 corrupt: 100%|██████████| 649/649 [00:00<?, ?it/s]

WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 742, len(boxes) = 3716. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 41/41 [00:10<00:00,  4.03it/s]


                   all        649       3716      0.537     0.0762      0.014     0.0067
          large_debris        141       3136      0.103    0.00478    0.00882    0.00349
         medium_debris        514        531          1          0    0.00247   0.000878
                rocket          9          9          1          0          0          0
             satellite         39         40     0.0448        0.3     0.0448     0.0224


/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.6ms preprocess, 10.8ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs/detect/val4
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c1e95c63220>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039